In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [91]:
PJME_data = pd.read_csv("PJME_phase2_preprocessed.csv",parse_dates=["Datetime"])

In [92]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  Day  Week  Month  DayOfWeek  IsWeekend  \
Datetime                                                                     
2002-01-08 01:00:00  29445.0     1    8     2      1          1          0   
2002-01-08 02:00:00  28670.0     2    8     2      1          1          0   
2002-01-08 03:00:00  28375.0     3    8     2      1          1          0   
2002-01-08 04:00:00  28542.0     4    8     2      1          1          0   
2002-01-08 05:00:00  29261.0     5    8     2      1          1          0   

                       Lag_1   Lag_24   Lag_48  Lag_168  Rolling_Mean_24  \
Datetime                                                                   
2002-01-08 01:00:00  31187.0  26862.0  27100.0  30393.0     33452.583333   
2002-01-08 02:00:00  29445.0  25976.0  26097.0  29265.0     33560.208333   
2002-01-08 03:00:00  28670.0  25641.0  25793.0  28357.0     33672.458333   
2002-01-08 04:00:00  28375.0  25666.0  25657.0  27899.0     

In [4]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)


print(train.index.min(), "to", train.index.max())
print(validation.index.min(), "to", validation.index.max())
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)
2002-01-08 01:00:00 to 2013-08-13 16:00:00
2013-08-13 17:00:00 to 2016-02-07 08:00:00
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [5]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_48",
    "Lag_168",
    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

target = "PJME_MW"

X_train = train[features].copy()
X_val = validation[features].copy()
X_test = test[features].copy()

y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (101656, 15)
X_val: (21784, 15)
X_test: (21784, 15)


In [6]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val_scaled = target_scaler.transform(y_val.values.reshape(-1, 1))
y_test_scaled = target_scaler.transform(y_test.values.reshape(-1, 1))

print("Scaling completed.")

Scaling completed.


In [7]:
def create_sequences(X, y, input_steps=48, output_steps=24):

    X_seq = []
    y_seq = []

    for i in range(input_steps, len(X) - output_steps + 1):

        X_seq.append(X[i-input_steps:i])
        y_seq.append(y[i:i+output_steps].flatten())

    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=48,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=48,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=48,
    output_steps=24
)


print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101585, 48, 15)
y_train: (101585, 24)
X_val: (21713, 48, 15)
y_val: (21713, 24)
X_test: (21713, 48, 15)
y_test: (21713, 24)


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [9]:
gru_model_es = Sequential([
    GRU(64, input_shape=(48, 15), return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_model_es.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_model_es.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [12]:
gru_history_es = gru_model_es.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - loss: 0.1173 - mae: 0.2430 - val_loss: 0.0869 - val_mae: 0.2120
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0716 - mae: 0.1918 - val_loss: 0.0796 - val_mae: 0.2030
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0659 - mae: 0.1827 - val_loss: 0.0813 - val_mae: 0.2029
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0625 - mae: 0.1770 - val_loss: 0.0789 - val_mae: 0.2018
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.0600 - mae: 0.1731 - val_loss: 0.0765 - val_mae: 0.1976
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0583 - mae: 0.1703 - val_loss: 0.0721 - val_mae: 0.1909
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0565 - mae: 0.1676 - val_loss: 0.0738 - val_mae: 0.1914
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0549 - mae: 0.1652 - val_loss: 0.0755 - val_mae: 0.1964
Epoch 9/15
1588/1588 ━━━━━

In [13]:
gru_model_es_pred_scaled = gru_model_es.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [14]:
gru_pred = target_scaler.inverse_transform(
    gru_model_es_pred_scaled.reshape(-1, 1)
).reshape(gru_model_es_pred_scaled.shape)

gru_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [15]:
from sklearn.metrics import ( mean_absolute_error, mean_squared_error, r2_score)

gru_mae = mean_absolute_error(gru_actual.flatten(), gru_pred.flatten())
gru_rmse = np.sqrt(mean_squared_error(gru_actual.flatten(), gru_pred.flatten()))
gru_mape = np.mean(np.abs((gru_actual.flatten() - gru_pred.flatten()) / gru_actual.flatten())) * 100
gru_r2 = r2_score(gru_actual.flatten(), gru_pred.flatten())
gru_bias = np.mean( gru_pred.flatten() - gru_actual.flatten())


print("GRU + Earlystopping")
print("MAE :", gru_mae)
print("RMSE:", gru_rmse)
print("MAPE:", gru_mape)
print("R²  :", gru_r2)
print("Bias:", gru_bias)

GRU + Earlystopping
MAE : 1378.1796126937743
RMSE: 1931.7571797210753
MAPE: 4.370879419177679
R²  : 0.9101446722791471
Bias: 178.30436501129435


In [17]:
print("Epochs completed:", len(gru_history_es.history["loss"]))

Epochs completed: 9


**Dropout**

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

gru_dropout_model = Sequential([
    GRU(64, input_shape=(48, 15), return_sequences=False),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(24)
])

gru_dropout_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_dropout_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
gru_history_dropout = gru_dropout_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 22s 11ms/step - loss: 0.1935 - mae: 0.3289 - val_loss: 0.0955 - val_mae: 0.2276
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1218 - mae: 0.2635 - val_loss: 0.0859 - val_mae: 0.2148
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1137 - mae: 0.2539 - val_loss: 0.0840 - val_mae: 0.2115
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.1102 - mae: 0.2495 - val_loss: 0.0852 - val_mae: 0.2172
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.1074 - mae: 0.2463 - val_loss: 0.0787 - val_mae: 0.2030
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1051 - mae: 0.2437 - val_loss: 0.0780 - val_mae: 0.2011
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.1035 - mae: 0.2418 - val_loss: 0.0790 - val_mae: 0.2062
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.1023 - mae: 0.2402 - val_loss: 0.0775 - val_mae: 0.2006
Epoch 9/15
1588/1588 ━━━━

In [20]:
dropout_pred_scaled = gru_dropout_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [21]:
dropout_pred = target_scaler.inverse_transform(
    dropout_pred_scaled.reshape(-1, 1)
).reshape(dropout_pred_scaled.shape)

dropout_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [22]:
mae = mean_absolute_error(dropout_actual.flatten(), dropout_pred.flatten())
rmse = np.sqrt(mean_squared_error(dropout_actual.flatten(), dropout_pred.flatten()))
mape = np.mean(
    np.abs(
        (dropout_actual.flatten() - dropout_pred.flatten())
        / dropout_actual.flatten())) * 100
r2 = r2_score(dropout_actual.flatten(), dropout_pred.flatten())
bias = np.mean(dropout_pred.flatten() - dropout_actual.flatten())

print("== GRU + DROPOUT ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== GRU + DROPOUT ==
MAE : 1469.7903004888765
RMSE: 2015.4369185481455
MAPE: 4.708082386955354
R²  : 0.902191368675151
Bias: 276.50957978818013


**Batch Normalization**

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import  BatchNormalization

gru_bn_model = Sequential([
    GRU(64, input_shape=(48, 15)),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(24)
])

gru_bn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_bn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,784 (85.09 KB)

 Trainable params: 21,528 (84.09 KB)

 Non-trainable params: 256 (1.00 KB)

In [24]:
history_bn = gru_bn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - loss: 0.1683 - mae: 0.3017 - val_loss: 0.0979 - val_mae: 0.2326
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 32s 20ms/step - loss: 0.0977 - mae: 0.2349 - val_loss: 0.0903 - val_mae: 0.2250
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0891 - mae: 0.2233 - val_loss: 0.0830 - val_mae: 0.2067
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0848 - mae: 0.2173 - val_loss: 0.0797 - val_mae: 0.2040
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0807 - mae: 0.2117 - val_loss: 0.0802 - val_mae: 0.2025
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - loss: 0.0783 - mae: 0.2088 - val_loss: 0.0802 - val_mae: 0.2027
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0764 - mae: 0.2066 - val_loss: 0.0748 - val_mae: 0.1969
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 30s 19ms/step - loss: 0.0739 - mae: 0.2031 - val_loss: 0.0794 - val_mae: 0.1984
Epoch 9/15
1588/1588 ━━━

In [26]:
bn_pred_scaled = gru_bn_model.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [28]:
rnn_bn_pred = target_scaler.inverse_transform(
    bn_pred_scaled.reshape(-1, 1)
).reshape(bn_pred_scaled.shape)

rnn_bn_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [29]:
mae = mean_absolute_error(rnn_bn_actual.flatten(), rnn_bn_pred.flatten())
rmse = np.sqrt(mean_squared_error(rnn_bn_actual.flatten(), rnn_bn_pred.flatten()))
mape = np.mean(np.abs((rnn_bn_actual.flatten() - rnn_bn_pred.flatten()) / rnn_bn_actual.flatten())) * 100
r2 = r2_score(rnn_bn_actual.flatten(), rnn_bn_pred.flatten())
bias = np.mean(rnn_bn_pred.flatten() - rnn_bn_actual.flatten())

print("== GRU + Batch ==")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

== GRU + Batch ==
MAE : 1470.7275540919586
RMSE: 2094.037658914477
MAPE: 4.618978710054963
R²  : 0.8944136595470149
Bias: 194.20284623165222


**Adam**

In [30]:
from tensorflow.keras.optimizers import Adam

gru_adam = Sequential([
    GRU(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_adam.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [31]:
history_adam = gru_adam.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.1189 - mae: 0.2464 - val_loss: 0.0882 - val_mae: 0.2171
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0719 - mae: 0.1921 - val_loss: 0.0862 - val_mae: 0.2112
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0667 - mae: 0.1835 - val_loss: 0.0802 - val_mae: 0.2021
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0632 - mae: 0.1777 - val_loss: 0.0793 - val_mae: 0.2019
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0604 - mae: 0.1730 - val_loss: 0.0742 - val_mae: 0.1942
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 15ms/step - loss: 0.0581 - mae: 0.1696 - val_loss: 0.0738 - val_mae: 0.1922
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0564 - mae: 0.1670 - val_loss: 0.0738 - val_mae: 0.1928
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0545 - mae: 0.1641 - val_loss: 0.0770 - val_mae: 0.1939
Epoch 9/15
1588/1588 ━━━━━━━

In [32]:
adam_pred_scaled = gru_adam.predict(
    X_test_seq,
    batch_size=64
)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [33]:
adam_pred = target_scaler.inverse_transform(
    adam_pred_scaled.reshape(-1, 1)
).reshape(adam_pred_scaled.shape)

adam_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [34]:
adam_mae = mean_absolute_error(adam_actual.flatten(), adam_pred.flatten())
adam_rmse = np.sqrt(mean_squared_error(adam_actual.flatten(), adam_pred.flatten()))
adam_mape = np.mean(np.abs((adam_actual.flatten() - adam_pred.flatten()) / adam_actual.flatten())) * 100
adam_r2 = r2_score(adam_actual.flatten(), adam_pred.flatten())
adam_bias = np.mean(adam_pred.flatten() - adam_actual.flatten())

print("========== gru + ADAM ==========")
print("MAE :", adam_mae)
print("RMSE:", adam_rmse)
print("MAPE:", adam_mape)
print("R²  :", adam_r2)
print("Bias:", adam_bias)

========== gru + ADAM ==========
MAE : 1468.911247579845
RMSE: 2107.4390922234265
MAPE: 4.632437997807455
R²  : 0.8930578709510623
Bias: 265.1317132319024


**RMSPROP**

In [35]:
from tensorflow.keras.optimizers import RMSprop

gru_rmsprop = Sequential([
    GRU(64, input_shape=(48, 15), return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_rmsprop.compile(
    optimizer=RMSprop(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

gru_rmsprop.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_4 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
history_rmsprop = gru_rmsprop.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.1193 - mae: 0.2497 - val_loss: 0.0973 - val_mae: 0.2274
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 8ms/step - loss: 0.0773 - mae: 0.2006 - val_loss: 0.0923 - val_mae: 0.2172
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0701 - mae: 0.1890 - val_loss: 0.0904 - val_mae: 0.2221
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0661 - mae: 0.1821 - val_loss: 0.0814 - val_mae: 0.2019
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.0632 - mae: 0.1775 - val_loss: 0.0796 - val_mae: 0.2038
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0610 - mae: 0.1739 - val_loss: 0.0894 - val_mae: 0.2083
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0589 - mae: 0.1707 - val_loss: 0.0794 - val_mae: 0.2019
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.0571 - mae: 0.1680 - val_loss: 0.0897 - val_mae: 0.2178
Epoch 9/15
1588/1588 ━━━━━━

In [37]:
rmsprop_pred_scaled = gru_rmsprop.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rmsprop_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [38]:
rmsprop_pred = target_scaler.inverse_transform(
    rmsprop_pred_scaled.reshape(-1, 1)
).reshape(rmsprop_pred_scaled.shape)

rmsprop_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [39]:
rmsprop_mae = mean_absolute_error(rmsprop_actual.flatten(), rmsprop_pred.flatten())
rmsprop_rmse = np.sqrt(mean_squared_error(rmsprop_actual.flatten(), rmsprop_pred.flatten()))
rmsprop_mape = np.mean(np.abs((rmsprop_actual.flatten() - rmsprop_pred.flatten())/ rmsprop_actual.flatten()) * 100
rmsprop_r2 = r2_score(rmsprop_actual.flatten(), rmsprop_pred.flatten())
rmsprop_bias = np.mean(rmsprop_pred.flatten() - rmsprop_actual.flatten())


print("=== GRU + RMSPROP ===")
print("MAE :", rmsprop_mae)
print("RMSE:", rmsprop_rmse)
print("MAPE:", rmsprop_mape)
print("R²  :", rmsprop_r2)
print("Bias:", rmsprop_bias)

=== GRU + RMSPROP ===
MAE : 1469.8973783841693
RMSE: 2105.9670044145905
MAPE: 4.616610547311478
R²  : 0.8932072211473744
Bias: 125.54715442799484


**SGD**

In [40]:
from tensorflow.keras.optimizers import SGD

gru_sgd = Sequential([
    GRU(64, input_shape=(48, 15), return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_sgd.compile(
    optimizer=SGD(
        learning_rate=0.001,
        momentum=0.9
    ),
    loss="mse",
    metrics=["mae"]
)

gru_sgd.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_5 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,272 (83.09 KB)

 Trainable params: 21,272 (83.09 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
history_sgd = gru_sgd.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.5155 - mae: 0.5581 - val_loss: 0.2994 - val_mae: 0.4364
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.2274 - mae: 0.3778 - val_loss: 0.1910 - val_mae: 0.3451
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 0.1731 - mae: 0.3266 - val_loss: 0.1678 - val_mae: 0.3194
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1535 - mae: 0.3041 - val_loss: 0.1543 - val_mae: 0.3043
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.1414 - mae: 0.2895 - val_loss: 0.1462 - val_mae: 0.2948
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - loss: 0.1331 - mae: 0.2790 - val_loss: 0.1395 - val_mae: 0.2866
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1267 - mae: 0.2708 - val_loss: 0.1344 - val_mae: 0.2804
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.1215 - mae: 0.2639 - val_loss: 0.1293 - val_mae: 0.2734
Epoch 9/15
1588/1588 ━━━━

In [42]:
sgd_pred_scaled = gru_sgd.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", sgd_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [43]:
sgd_pred = target_scaler.inverse_transform(
    sgd_pred_scaled.reshape(-1, 1)
).reshape(sgd_pred_scaled.shape)

sgd_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [44]:
sgd_mae = mean_absolute_error(sgd_actual.flatten(), sgd_pred.flatten())
sgd_rmse = np.sqrt(mean_squared_error(sgd_actual.flatten(), sgd_pred.flatten()))
sgd_mape = np.mean(np.abs((sgd_actual.flatten() - sgd_pred.flatten()) / sgd_actual.flatten())) * 100
sgd_r2 = r2_score(sgd_actual.flatten(), sgd_pred.flatten())
sgd_bias = np.mean(sgd_pred.flatten() - sgd_actual.flatten())


print("=== GRU + SGD ===")
print("MAE :", sgd_mae)
print("RMSE:", sgd_rmse)
print("MAPE:", sgd_mape)
print("R²  :", sgd_r2)
print("Bias:", sgd_bias)

=== GRU + SGD ===
MAE : 1694.3271390281593
RMSE: 2290.7248902820465
MAPE: 5.43652675089115
R²  : 0.8736472693917308
Bias: 147.16616536675718


Learning Rate Scheduling

In [45]:
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import RMSprop

gru_lr = Sequential([
    GRU(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_lr.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [46]:
history_lr = gru_lr.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.1182 - mae: 0.2471 - val_loss: 0.0945 - val_mae: 0.2249 - learning_rate: 0.0010
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0755 - mae: 0.1970 - val_loss: 0.0865 - val_mae: 0.2102 - learning_rate: 0.0010
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0692 - mae: 0.1868 - val_loss: 0.0881 - val_mae: 0.2125 - learning_rate: 0.0010
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0654 - mae: 0.1809 - val_loss: 0.0828 - val_mae: 0.2049 - learning_rate: 0.0010
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - loss: 0.0625 - mae: 0.1764 - val_loss: 0.0816 - val_mae: 0.2027 - learning_rate: 0.0010
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0604 - mae: 0.1731 - val_loss: 0.0801 - val_mae: 0.2018 - learning_rate: 0.0010
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0585 - mae: 0.1703 - val_loss: 0.0787 - val_mae: 0.2044 - learni

In [47]:
lr_pred_scaled = gru_lr.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", lr_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [48]:
lr_pred = target_scaler.inverse_transform(
    lr_pred_scaled.reshape(-1, 1)
).reshape(lr_pred_scaled.shape)

lr_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [49]:
lr_mae = mean_absolute_error(lr_actual.flatten(), lr_pred.flatten())
lr_rmse = np.sqrt(mean_squared_error(lr_actual.flatten(),lr_pred.flatten()))
lr_mape = np.mean(np.abs((lr_actual.flatten() - lr_pred.flatten()) / lr_actual.flatten())) * 100
lr_r2 = r2_score(lr_actual.flatten(), lr_pred.flatten())
lr_bias = np.mean(lr_pred.flatten() - lr_actual.flatten())


print("==== GRU + RMSPROP + LearningRate ====")
print("MAE :", lr_mae)
print("RMSE:", lr_rmse)
print("MAPE:", lr_mape)
print("R²  :", lr_r2)
print("Bias:", lr_bias)

==== GRU + RMSPROP + LearningRate ====
MAE : 1414.370091791088
RMSE: 2034.2995105844225
MAPE: 4.472339894385085
R²  : 0.9003520080178874
Bias: 288.2456998132719


**Layers**

In [50]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

gru_deep = Sequential([
    GRU(64, return_sequences=True, input_shape=(48, 15)),
    GRU(32, return_sequences=False),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_deep.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_deep.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_7 (GRU)                     │ (None, 48, 64)         │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_8 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,632 (111.84 KB)

 Trainable params: 28,632 (111.84 KB)

 Non-trainable params: 0 (0.00 B)

In [51]:
history_deep = gru_deep.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.1189 - mae: 0.2440 - val_loss: 0.0919 - val_mae: 0.2226
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0701 - mae: 0.1901 - val_loss: 0.0810 - val_mae: 0.2056
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0646 - mae: 0.1811 - val_loss: 0.0788 - val_mae: 0.1987
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0607 - mae: 0.1749 - val_loss: 0.0757 - val_mae: 0.1957
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0581 - mae: 0.1709 - val_loss: 0.0755 - val_mae: 0.1944
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 23s 15ms/step - loss: 0.0554 - mae: 0.1668 - val_loss: 0.0751 - val_mae: 0.1956
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 42s 15ms/step - loss: 0.0529 - mae: 0.1633 - val_loss: 0.0792 - val_mae: 0.1995
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0503 - mae: 0.1598 - val_loss: 0.0777 - val_mae: 0.1976
Epoch 9/15
1588/1588 ━━━━━━

In [52]:
deep_pred_scaled = gru_deep.predict(
    X_test_seq,
    batch_size=64
)
print("Prediction shape:", deep_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Prediction shape: (21713, 24)


In [53]:
deep_pred = target_scaler.inverse_transform(
    deep_pred_scaled.reshape(-1, 1)
).reshape(deep_pred_scaled.shape)

deep_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [54]:
deep_mae = mean_absolute_error(deep_actual.flatten(), deep_pred.flatten())
deep_rmse = np.sqrt(mean_squared_error(deep_actual.flatten(), deep_pred.flatten()))
deep_mape = np.mean(np.abs((deep_actual.flatten() - deep_pred.flatten()) / deep_actual.flatten())) * 100
deep_r2 = r2_score(deep_actual.flatten(), deep_pred.flatten())
deep_bias = np.mean(deep_pred.flatten() - deep_actual.flatten())

print("==== GRU + Layers ==========")
print("MAE :", deep_mae)
print("RMSE:", deep_rmse)
print("MAPE:", deep_mape)
print("R²  :", deep_r2)
print("Bias:", deep_bias)

==== GRU + Layers ==========
MAE : 1524.688810823657
RMSE: 2205.6863149204455
MAPE: 4.782851174745032
R²  : 0.8828543248625991
Bias: 235.1875096548343


**Neurons**

In [55]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

gru_128 = Sequential([
    GRU(128, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_128.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_9 (GRU)                     │ (None, 128)            │        55,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,496 (255.84 KB)

 Trainable params: 65,496 (255.84 KB)

 Non-trainable params: 0 (0.00 B)

In [56]:
history_128 = gru_128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.1057 - mae: 0.2307 - val_loss: 0.0902 - val_mae: 0.2201
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0683 - mae: 0.1864 - val_loss: 0.0829 - val_mae: 0.2083
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0625 - mae: 0.1771 - val_loss: 0.0774 - val_mae: 0.1983
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0580 - mae: 0.1698 - val_loss: 0.0763 - val_mae: 0.1953
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0547 - mae: 0.1650 - val_loss: 0.0755 - val_mae: 0.1933
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0509 - mae: 0.1598 - val_loss: 0.0847 - val_mae: 0.2015
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0464 - mae: 0.1537 - val_loss: 0.0790 - val_mae: 0.1982
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0411 - mae: 0.1467 - val_loss: 0.0830 - val_mae: 0.1990
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [57]:
rnn128_pred_scaled = gru_128.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rnn128_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Prediction shape: (21713, 24)


In [58]:
rnn128_pred = target_scaler.inverse_transform(
    rnn128_pred_scaled.reshape(-1, 1)
).reshape(rnn128_pred_scaled.shape)

rnn128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [59]:
rnn128_mae = mean_absolute_error(gru128_actual.flatten(), gru128_pred.flatten())
rnn128_rmse = np.sqrt(mean_squared_error(gru128_actual.flatten(), gru128_pred.flatten()))
rnn128_mape = np.mean(np.abs((gru128_actual.flatten() - gru128_pred.flatten()) / gru128_actual.flatten())) * 100
rnn128_r2 = r2_score(gru128_actual.flatten(), gru128_pred.flatten())
rnn128_bias = np.mean(gru128_pred.flatten() - gru128_actual.flatten())

print("=== GRU 128 NEURONS ===")
print("MAE :", rnn128_mae)
print("RMSE:", rnn128_rmse)
print("MAPE:", rnn128_mape)
print("R²  :", rnn128_r2)
print("Bias:", rnn128_bias)

=== GRU 128 NEURONS ===
MAE : 1565.1367326213283
RMSE: 2258.028879185362
MAPE: 4.913363297377043
R²  : 0.8772284478461471
Bias: 270.10715336491364


32 **neurons**

In [60]:
gru_32 = Sequential([
    GRU(32, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_32.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_10 (GRU)                    │ (None, 32)             │         4,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,376 (32.72 KB)

 Trainable params: 8,376 (32.72 KB)

 Non-trainable params: 0 (0.00 B)

In [61]:
history_32 = gru_32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.1439 - mae: 0.2699 - val_loss: 0.0938 - val_mae: 0.2234
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0760 - mae: 0.1990 - val_loss: 0.0839 - val_mae: 0.2089
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0701 - mae: 0.1891 - val_loss: 0.0841 - val_mae: 0.2066
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0667 - mae: 0.1835 - val_loss: 0.0805 - val_mae: 0.2044
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0646 - mae: 0.1801 - val_loss: 0.0780 - val_mae: 0.1986
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0627 - mae: 0.1768 - val_loss: 0.0834 - val_mae: 0.2052
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0615 - mae: 0.1748 - val_loss: 0.0756 - val_mae: 0.1947
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - loss: 0.0603 - mae: 0.1729 - val_loss: 0.0787 - val_mae: 0.1997
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [62]:
rnn32_pred_scaled = gru_32.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", rnn32_pred_scaled.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [63]:
rnn32_pred = target_scaler.inverse_transform(
    rnn32_pred_scaled.reshape(-1, 1)
).reshape(rnn32_pred_scaled.shape)

rnn32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [64]:
rnn32_mae = mean_absolute_error(rnn32_actual.flatten(), rnn32_pred.flatten())
rnn32_rmse = np.sqrt(mean_squared_error(rnn32_actual.flatten(), rnn32_pred.flatten()))
rnn32_mape = np.mean(np.abs((rnn32_actual.flatten() - rnn32_pred.flatten()) / rnn32_actual.flatten())) * 100
rnn32_r2 = r2_score(rnn32_actual.flatten(), rnn32_pred.flatten())
rnn32_bias = np.mean(rnn32_pred.flatten() - rnn32_actual.flatten())

print("=== GRU 32 NEURONS ===")
print("MAE :", rnn32_mae)
print("RMSE:", rnn32_rmse)
print("MAPE:", rnn32_mape)
print("R²  :", rnn32_r2)
print("Bias:", rnn32_bias)

=== GRU 32 NEURONS ===
MAE : 1433.3135730358829
RMSE: 2030.0157667862943
MAPE: 4.5438640995965915
R²  : 0.9007712353996487
Bias: 343.8398419959385


Batch Size(32)

In [65]:
gru_batch32 = Sequential([
    GRU(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_batch32.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch32 = gru_batch32.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=32,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - loss: 0.1058 - mae: 0.2317 - val_loss: 0.0841 - val_mae: 0.2104
Epoch 2/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0696 - mae: 0.1882 - val_loss: 0.0835 - val_mae: 0.2076
Epoch 3/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step - loss: 0.0641 - mae: 0.1792 - val_loss: 0.0744 - val_mae: 0.1927
Epoch 4/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0606 - mae: 0.1737 - val_loss: 0.0797 - val_mae: 0.2031
Epoch 5/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0577 - mae: 0.1692 - val_loss: 0.0777 - val_mae: 0.1972
Epoch 6/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 0.0554 - mae: 0.1659 - val_loss: 0.0749 - val_mae: 0.1944
Epoch 7/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step - loss: 0.0530 - mae: 0.1628 - val_loss: 0.0753 - val_mae: 0.1933
Epoch 8/15
3175/3175 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - loss: 0.0506 - mae: 0.1597 - val_loss: 0.0778 - val_mae: 0.2014
Epoch 9/15
3175/3175 ━━━━━━━━━━━

In [66]:
rnn_batch32_pred_scaled = gru_batch32.predict(
    X_test_seq,
    batch_size=32
)

print("Prediction shape:", rnn_batch32_pred_scaled.shape)

679/679 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Prediction shape: (21713, 24)


In [67]:
rnnbatch32_pred = target_scaler.inverse_transform(
    rnn_batch32_pred_scaled.reshape(-1, 1)
).reshape(rnn_batch32_pred_scaled.shape)

rnnbatch32_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [68]:
rnnb32_mae = mean_absolute_error(rnnbatch32_actual.flatten(), rnnbatch32_pred.flatten())
rnnb32_rmse = np.sqrt(mean_squared_error(rnnbatch32_actual.flatten(), rnnbatch32_pred.flatten()))
rnnb32_mape = np.mean(np.abs((rnnbatch32_actual.flatten() - rnnbatch32_pred.flatten()) / rnnbatch32_actual.flatten())) * 100
rnnb32_r2 = r2_score(rnnbatch32_actual.flatten(), rnnbatch32_pred.flatten())
rnnb32_bias = np.mean(rnnbatch32_pred.flatten() - rnnbatch32_actual.flatten())

print("=== GRU batch32 NEURONS ===")
print("MAE :", rnnb32_mae)
print("RMSE:", rnnb32_rmse)
print("MAPE:", rnnb32_mape)
print("R²  :", rnnb32_r2)
print("Bias:", rnnb32_bias)

=== GRU batch32 NEURONS ===
MAE : 1518.474878192422
RMSE: 2188.7714646917384
MAPE: 4.775740437993456
R²  : 0.8846441566437921
Bias: 273.57027589333603


BatchSize(128)

In [76]:
gru_batch128 = Sequential([
    GRU(64, input_shape=(48, 15)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_batch128.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_batch128 = gru_batch128.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=15,
    batch_size=128,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.1374 - mae: 0.2629 - val_loss: 0.0932 - val_mae: 0.2234
Epoch 2/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - loss: 0.0740 - mae: 0.1955 - val_loss: 0.0834 - val_mae: 0.2056
Epoch 3/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0682 - mae: 0.1860 - val_loss: 0.0800 - val_mae: 0.2024
Epoch 4/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0651 - mae: 0.1807 - val_loss: 0.0807 - val_mae: 0.2024
Epoch 5/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0631 - mae: 0.1773 - val_loss: 0.0785 - val_mae: 0.2004
Epoch 6/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0613 - mae: 0.1745 - val_loss: 0.0749 - val_mae: 0.1949
Epoch 7/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0596 - mae: 0.1717 - val_loss: 0.0752 - val_mae: 0.1961
Epoch 8/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.0584 - mae: 0.1697 - val_loss: 0.0755 - val_mae: 0.1932
Epoch 9/15
794/794 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - lo

In [77]:
gru_batch128_pred_scaled = gru_batch128.predict(
    X_test_seq,
    batch_size=128
)

print("Prediction shape:", gru_batch128_pred_scaled.shape)

170/170 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Prediction shape: (21713, 24)


In [79]:
rnnbatch128_pred = target_scaler.inverse_transform(
    gru_batch128_pred_scaled.reshape(-1, 1)
).reshape(gru_batch128_pred_scaled.shape)

rnnbatch128_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [80]:
rnnb128_mae = mean_absolute_error(rnnbatch128_actual.flatten(), rnnbatch128_pred.flatten())
rnnb128_rmse = np.sqrt(mean_squared_error(rnnbatch128_actual.flatten(), rnnbatch128_pred.flatten()))
rnnb128_mape = np.mean(np.abs((rnnbatch128_actual.flatten() - rnnbatch128_pred.flatten())  / rnnbatch128_actual.flatten())) * 100
rnnb128_r2 = r2_score(rnnbatch128_actual.flatten(), rnnbatch128_pred.flatten())
rnnb128_bias = np.mean(rnnbatch128_pred.flatten() - rnnbatch128_actual.flatten())

print("=== GRU batch128  ===")
print("MAE :", rnnb128_mae)
print("RMSE:", rnnb128_rmse)
print("MAPE:", rnnb128_mape)
print("R²  :", rnnb128_r2)
print("Bias:", rnnb128_bias)

=== GRU batch128  ===
MAE : 1415.9205237692124
RMSE: 2017.9648071192717
MAPE: 4.4678725254104625
R²  : 0.9019458592504876
Bias: 198.01213071833047


**HyperParameter Tuning**

In [81]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.2 MB/s eta 0:00:00


In [82]:
import keras_tuner as kt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [83]:
def build_gru(hp):

    model = Sequential()
    model.add(
        GRU(
            units=hp.Choice(
                "rnn_units",
                values=[32, 64, 128]
            ),
            input_shape=(48, 15)
        )
    )
    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )
    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )
    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [84]:
tuner = kt.RandomSearch(
    build_gru,
    objective="val_loss",
    max_trials=6,
    executions_per_trial=1,
    directory="hyperparameter_tuning",
    project_name="gru_48_to_24"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [85]:
tuner.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 6 Complete [00h 03m 13s]
val_loss: 0.08781477808952332

Best val_loss So Far: 0.07114782184362411
Total elapsed time: 00h 19m 12s


In [86]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("BEST HYPERPARAMETERS")
print("RNN Units:",best_hp.get("rnn_units"))
print("Dense Units:",best_hp.get("dense_units"))
print("Learning Rate:",best_hp.get("learning_rate"))

BEST HYPERPARAMETERS
RNN Units: 128
Dense Units: 128
Learning Rate: 0.0005


In [87]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 128)            │        55,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,288 (294.09 KB)

 Trainable params: 75,288 (294.09 KB)

 Non-trainable params: 0 (0.00 B)

In [88]:
history_best = best_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.0554 - mae: 0.1648 - val_loss: 0.0737 - val_mae: 0.1901
Epoch 2/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0535 - mae: 0.1620 - val_loss: 0.0748 - val_mae: 0.1937
Epoch 3/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0517 - mae: 0.1594 - val_loss: 0.0716 - val_mae: 0.1866
Epoch 4/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - loss: 0.0497 - mae: 0.1565 - val_loss: 0.0800 - val_mae: 0.1954
Epoch 5/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0473 - mae: 0.1533 - val_loss: 0.0795 - val_mae: 0.2000
Epoch 6/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0447 - mae: 0.1499 - val_loss: 0.0769 - val_mae: 0.1931
Epoch 7/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0420 - mae: 0.1462 - val_loss: 0.0800 - val_mae: 0.1982
Epoch 8/15
1588/1588 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0391 - mae: 0.1421 - val_loss: 0.0804 - val_mae: 0.1953
Epoch 9/15
1588/1588 ━━━━━━━━━━━

In [89]:
best_pred_scaled = best_model.predict(
    X_test_seq,
    batch_size=64
)

best_pred = target_scaler.inverse_transform(
    best_pred_scaled.reshape(-1, 1)
).reshape(best_pred_scaled.shape)

best_actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

340/340 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [90]:
best_mae = mean_absolute_error(best_actual.flatten(), best_pred.flatten())
best_rmse = np.sqrt(mean_squared_error(best_actual.flatten(), best_pred.flatten()))
best_mape = np.mean(np.abs((best_actual.flatten() - best_pred.flatten()) / best_actual.flatten())) * 100
best_r2 = r2_score(best_actual.flatten(), best_pred.flatten())
best_bias = np.mean(best_pred.flatten() - best_actual.flatten())

print("HYPERPARAMETER TUNED GRU")
print("MAE :", best_mae)
print("RMSE:", best_rmse)
print("MAPE:", best_mape)
print("R²  :", best_r2)
print("Bias:", best_bias)

HYPERPARAMETER TUNED GRU
MAE : 1547.62501968072
RMSE: 2277.535132282997
MAPE: 4.865734143585166
R²  : 0.8750981323917247
Bias: 331.7982502179534


**Additional Logs**

In [96]:
# Additional lag features

PJME_data["Lag_2"] = PJME_data["PJME_MW"].shift(2)
PJME_data["Lag_3"] = PJME_data["PJME_MW"].shift(3)
PJME_data["Lag_6"] = PJME_data["PJME_MW"].shift(6)
PJME_data["Lag_12"] = PJME_data["PJME_MW"].shift(12)
PJME_data["Lag_72"] = PJME_data["PJME_MW"].shift(72)
PJME_data["Lag_336"] = PJME_data["PJME_MW"].shift(336)

In [97]:
PJME_data = PJME_data.dropna().copy()

print("Missing values:")
print(PJME_data.isnull().sum().sum())

print("Shape:")
print(PJME_data.shape)

Missing values:
0
Shape:
(144552, 21)


In [98]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",

    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_24",
    "Lag_48",
    "Lag_72",
    "Lag_168",
    "Lag_336",

    "Rolling_Mean_24",
    "Rolling_Mean_168",
    "Rolling_Std_24",
    "Rolling_Std_168"
]

print("Number of features:", len(features))

Number of features: 21


In [99]:
n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = PJME_data.iloc[:train_end]
val_data = PJME_data.iloc[train_end:val_end]
test_data = PJME_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

Train: (101186, 21)
Validation: (21683, 21)
Test: (21683, 21)


In [100]:
X_train = train_data[features]
y_train = train_data[["PJME_MW"]]

X_val = val_data[features]
y_val = val_data[["PJME_MW"]]

X_test = test_data[features]
y_test = test_data[["PJME_MW"]]

In [101]:
from sklearn.preprocessing import StandardScaler

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)
X_val_scaled = feature_scaler.transform(X_val)
X_test_scaled = feature_scaler.transform(X_test)

y_train_scaled = target_scaler.fit_transform(y_train)
y_val_scaled = target_scaler.transform(y_val)
y_test_scaled = target_scaler.transform(y_test)

print("Scaling completed.")

Scaling completed.


In [102]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    input_steps=48,
    output_steps=24
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    input_steps=48,
    output_steps=24
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    input_steps=48,
    output_steps=24
)

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

X_train: (101115, 48, 21)
y_train: (101115, 24)
X_val: (21612, 48, 21)
y_val: (21612, 24)
X_test: (21612, 48, 21)
y_test: (21612, 24)


In [103]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense

gru_lags = Sequential([
    GRU(64, input_shape=(48, 21)),
    Dense(64, activation="relu"),
    Dense(24)
])

gru_lags.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

gru_lags.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 64)             │        16,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,424 (87.59 KB)

 Trainable params: 22,424 (87.59 KB)

 Non-trainable params: 0 (0.00 B)

In [104]:
history_lags = gru_lags.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Epoch 1/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.1185 - mae: 0.2433 - val_loss: 0.0875 - val_mae: 0.2137
Epoch 2/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0710 - mae: 0.1905 - val_loss: 0.0821 - val_mae: 0.2069
Epoch 3/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0648 - mae: 0.1803 - val_loss: 0.0802 - val_mae: 0.2019
Epoch 4/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0613 - mae: 0.1745 - val_loss: 0.0747 - val_mae: 0.1954
Epoch 5/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0589 - mae: 0.1706 - val_loss: 0.0736 - val_mae: 0.1924
Epoch 6/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0567 - mae: 0.1672 - val_loss: 0.0723 - val_mae: 0.1911
Epoch 7/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0547 - mae: 0.1644 - val_loss: 0.0741 - val_mae: 0.1917
Epoch 8/15
1580/1580 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0526 - mae: 0.1616 - val_loss: 0.0743 - val_mae: 0.1936
Epoch 9/15
1580/1580 ━━━━━━━━━━

In [105]:
pred_scaled = gru_lags.predict(
    X_test_seq,
    batch_size=64
)

print("Prediction shape:", pred_scaled.shape)

338/338 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Prediction shape: (21612, 24)


In [106]:
pred = target_scaler.inverse_transform(
    pred_scaled.reshape(-1, 1)
).reshape(pred_scaled.shape)

actual = target_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [107]:
mae = mean_absolute_error(actual.flatten(), pred.flatten())
rmse = np.sqrt(mean_squared_error(actual.flatten(), pred.flatten()))
mape = np.mean(np.abs((actual.flatten() - pred.flatten()) / actual.flatten())) * 100
r2 = r2_score(actual.flatten(), pred.flatten())
bias = np.mean(pred.flatten() - actual.flatten())

print("=== GRU + ADDITIONAL LAGS ===")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R²  :", r2)
print("Bias:", bias)

=== GRU + ADDITIONAL LAGS ===
MAE : 1494.5324586949127
RMSE: 2113.0760176594413
MAPE: 4.7341163414849285
R²  : 0.8924115137690171
Bias: 307.8776334359781


In [1]:
import pandas as pd

gru_results = [
    ["GRU Baseline", 48, 24, 1408.3473658666167, 2007.9125428794946, 4.42447204482822, 0.902920317414255, 130.56038532518562],
    ["GRU + EarlyStopping", 48, 24, 1378.1796126937743, 1931.7571797210753, 4.370879419177679, 0.9101446722791471, 178.30436501129435],
    ["GRU + Dropout", 48, 24, 1469.7903004888765, 2015.4369185481455, 4.708082386955354, 0.902191368675151, 276.50957978818013],
    ["GRU + Batch Normalization", 48, 24, 1470.7275540919586, 2094.037658914477, 4.618978710054963, 0.8944136595470149, 194.20284623165222],
    ["GRU + Adam", 48, 24, 1468.9112475798453, 2107.4390922234265, 4.632437997807455, 0.8930578709510623, 265.1317132319024],
    ["GRU + RMSprop", 48, 24, 1469.8973783841693, 2105.9670044145905, 4.616610547311478, 0.8932072211473744, 125.54715442799484],
    ["GRU + SGD", 48, 24, 1694.3271390281593, 2290.7248902820465, 5.43652675089115, 0.8736472693917308, 147.16616536675718],
    ["GRU + RMSprop + Learning Rate", 48, 24, 1414.370091791088, 2034.2995105844225, 4.472339894385085, 0.9003520080178874, 288.2456998132719],
    ["GRU + Additional Layers", 48, 24, 1524.688810823657, 2205.6863149204455, 4.782851174745032, 0.8828543248625991, 235.1875096548343],
    ["GRU 128 Neurons", 48, 24, 1565.1367326213283, 2258.028879185362, 4.913363297377043, 0.8772284478461471, 270.10715336491364],
    ["GRU 32 Neurons", 48, 24, 1433.3135730358829, 2030.0157667862943, 4.5438640995965915, 0.9007712353996487, 343.8398419959385],
    ["GRU Batch Size 32", 48, 24, 1518.474878192422, 2188.7714646917384, 4.775740437993456, 0.8846441566437921, 273.57027589333603],
    ["GRU Batch Size 128", 48, 24, 1415.9205237692124, 2017.9648071192717, 4.4678725254104625, 0.9019458592504876, 198.01213071833047],
    ["GRU Hyperparameter Tuned", 48, 24, 1547.62501968072, 2277.535132282997, 4.865734143585166, 0.8750981323917247, 331.7982502179534],
    ["GRU + Additional Lags", 48, 24, 1494.5324586949127, 2113.0760176594413, 4.7341163414849285, 0.8924115137690171, 307.8776334359781]
]

gru_results_df = pd.DataFrame(
    gru_results,
    columns=[
        "Model",
        "Input Hours",
        "Output Hours",
        "MAE",
        "RMSE",
        "MAPE (%)",
        "R²",
        "Bias"
    ]
)

# Round metrics
gru_results_df["MAE"] = gru_results_df["MAE"].round(6)
gru_results_df["RMSE"] = gru_results_df["RMSE"].round(6)
gru_results_df["MAPE (%)"] = gru_results_df["MAPE (%)"].round(6)
gru_results_df["R²"] = gru_results_df["R²"].round(6)
gru_results_df["Bias"] = gru_results_df["Bias"].round(6)

# Display
display(gru_results_df)


,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
0,GRU Baseline,48,24,1408.347366,2007.912543,4.424472,0.902920,130.560385
1,GRU + EarlyStopping,48,24,1378.179613,1931.757180,4.370879,0.910145,178.304365
2,GRU + Dropout,48,24,1469.790300,2015.436919,4.708082,0.902191,276.509580
3,GRU + Batch Normalization,48,24,1470.727554,2094.037659,4.618979,0.894414,194.202846
4,GRU + Adam,48,24,1468.911248,2107.439092,4.632438,0.893058,265.131713
5,GRU + RMSprop,48,24,1469.897378,2105.967004,4.616611,0.893207,125.547154
6,GRU + SGD,48,24,1694.327139,2290.724890,5.436527,0.873647,147.166165
7,GRU + RMSprop + Learning Rate,48,24,1414.370092,2034.299511,4.472340,0.900352,288.245700
8,GRU + Additional Layers,48,24,1524.688811,2205.686315,4.782851,0.882854,235.187510
9,GRU 128 Neurons,48,24,1565.136733,2258.028879,4.913363,0.877228,270.107153


In [111]:
import os

os.makedirs("comparison", exist_ok=True)

gru_results_df.to_csv(
    "comparison/GRU_48h_to_24h_results.csv",
    index=False,
    float_format="%.6f"
)

print("Saved inside comparison folder")

Saved inside comparison folder


In [2]:
best_gru_48_24 = gru_results_df.loc[gru_results_df["Model"] == "GRU + EarlyStopping"]
display(best_gru_48_24)

,Model,Input Hours,Output Hours,MAE,RMSE,MAPE (%),R²,Bias
1,GRU + EarlyStopping,48,24,1378.179613,1931.75718,4.370879,0.910145,178.304365
